
## 1. Setup and Dependencies

This section prepares the Python environment by installing and importing the libraries required for PDF text extraction, data processing, and validation.

**Libraries:** PyMuPDF, Pandas, and Regular Expressions.

**Expected output:** Successful environment initialization.



In [ ]:

%pip install -q pymupdf pandas

import re
from pathlib import Path

import fitz
import pandas as pd
from IPython.display import display

print("Origin | Data Preparation")
print("PyMuPDF version:", fitz.VersionBind)
print("Setup completed successfully!")


Origin | Data Preparation
PyMuPDF version: 1.28.2
Setup completed successfully!



## 2. Upload and Load PDF

Upload the original graduation projects PDF from Princess Nourah bint Abdulrahman University.

The document contains graduation project titles, abstracts, and related information from the 2024–2025 academic year.

This step verifies that the PDF is successfully loaded and reports its total number of pages.



In [ ]:

from google.colab import files

print("Upload the graduation projects PDF")

uploaded = files.upload()

assert len(uploaded) == 1, "Please upload one PDF file."

filename = next(iter(uploaded))

assert filename.lower().endswith(".pdf"), (
    "The uploaded file must be a PDF."
)

pdf_path = Path("/content") / filename
doc = fitz.open(str(pdf_path))

print("\nPDF INFORMATION")
print("=" * 45)
print("File:", filename)
print("Total pages:", len(doc))
print("PDF loaded successfully!")


Upload the graduation projects PDF


Saving Origin_Dataset.pdf to Origin_Dataset (1).pdf

PDF INFORMATION
File: Origin_Dataset (1).pdf
Total pages: 115
PDF loaded successfully!



## 3. PDF Inspection and Project Page Detection

Inspect the extracted text and identify pages containing graduation project abstracts.

Pages are classified using the standalone "Abstract" heading to distinguish actual project records from introductory pages, department covers, and other non-project content.

This ensures that only valid project pages are included in the dataset.



In [ ]:

# Inspect representative PDF pages

sample_pages = [1, 3, 5, 10, 26]

print("PDF TEXT INSPECTION")
print("=" * 55)

for page_num in sample_pages:
    text = doc[page_num - 1].get_text(
        "text", sort=True
    )

    print(f"\n--- PAGE {page_num} ---")
    print(text[:350])
    print("-" * 55)


# Identify actual project pages

abstract_pattern = re.compile(
    r"(?im)^[ \t]*Abstract[ \t]*$"
)

project_pages = []

for page_num, page in enumerate(doc, start=1):
    text = page.get_text("text", sort=True)

    if abstract_pattern.search(text):
        project_pages.append(page_num)


non_project_pages = [
    page_num
    for page_num in range(1, len(doc) + 1)
    if page_num not in project_pages
]

print("\nPAGE DETECTION SUMMARY")
print("=" * 55)

print("Total PDF pages:", len(doc))
print("Detected projects:", len(project_pages))
print("Non-project pages:", non_project_pages)


PDF TEXT INSPECTION

--- PAGE 1 ---
Princess Nourah Bint Abdulrahman University
College of Computer and Information Sciences





            Graduation Projects
           Book of Abstracts
               2024-2025
-------------------------------------------------------

--- PAGE 3 ---
Contents


Department of Computer Science - Bachelor of Science in Computer Science
.......................................................................................................................... 4

Department of Computer Science - Bachelor of Science in Artificial
Intelligence ............................................................
-------------------------------------------------------

--- PAGE 5 ---
              Automatic Plant Watering and Tracking System

  Refan Hussein BinMusaad, Rand Aiman Alsaigal, Hamdah Hamad Alanazi,
                      Lana Khalid Alrumaih

                       Supervisor: Dr. Ahmed Ibrahim

Abstract

Many people struggle with watering their plants reg


## 4. Initial Project Extraction

Extract the project header and abstract from each identified project page.

The header contains the project title, student names, and supervisor information. The abstract contains the project description and, where available, keywords.

The extracted information is stored in a preliminary DataFrame while preserving the original PDF page numbers.

**Output:** `raw_df`


In [ ]:

records = []

for page_num in project_pages:

    text = doc[page_num - 1].get_text(
        "text", sort=True
    )

    match = abstract_pattern.search(text)

    if match is None:
        continue

    header = text[:match.start()].strip()
    abstract = text[match.end():].strip()

    records.append({
        "source_page": page_num,
        "header_text": header,
        "abstract_text": abstract
    })


raw_df = pd.DataFrame(records)

raw_df = raw_df.sort_values(
    "source_page"
).reset_index(drop=True)

print("INITIAL EXTRACTION")
print("=" * 50)

print("Extracted records:", len(raw_df))
print(
    "Empty abstracts:",
    raw_df["abstract_text"].eq("").sum()
)

print("\nSAMPLE PROJECT HEADERS")

for _, row in raw_df.head(3).iterrows():
    print("\nPage:", row["source_page"])
    print(row["header_text"][:350])
    print("-" * 50)


INITIAL EXTRACTION
Extracted records: 103
Empty abstracts: 0

SAMPLE PROJECT HEADERS

Page: 5
Automatic Plant Watering and Tracking System

  Refan Hussein BinMusaad, Rand Aiman Alsaigal, Hamdah Hamad Alanazi,
                      Lana Khalid Alrumaih

                       Supervisor: Dr. Ahmed Ibrahim
--------------------------------------------------

Page: 6
CourseAssist: CLO Performance Assistant “E-CPA”

   Raghad Khalid Aldreweesh, Renad Abdulrahman Ajina, Shahad Ibrahim

                         Alharbi, Yara melaifi Alanazi

                       Supervisor: Dr. Ahmed Ibrahim
--------------------------------------------------

Page: 7
Mobile Application for Deaf and Mute individuals. (WASIF)

  Reemas turki alaqil, Abeer Sultan Alabdullatif, Haifa Abdulkarim Alkhalaf,

                       Layan Yahya Almegbil.

                  Supervisor: Dr. Eman Abdullah Aldakheel
--------------------------------------------------



## 5. Data Cleaning and Structured Dataset Creation

Transform the extracted information into a structured dataset.

The preprocessing includes:

- Extracting project titles from their headers.
- Removing repeated PDF footers and unnecessary whitespace.
- Separating keywords from project abstracts.
- Assigning the appropriate academic department.
- Generating unique project identifiers.
- Preserving the academic year and original source page.

A temporary review flag is included to identify potentially irregular title formatting.

**Output:** `projects_df`



In [ ]:

# Department page ranges


# Updated department ranges for the 115-page PDF

department_ranges = [
    (5, 37, "Computer Science"),
    (39, 50, "Artificial Intelligence"),
    (52, 74, "Information Systems"),
    (76, 85, "Data Science and Analysis"),
    (87, 103, "Information Technology"),
    (105, 112, "Cybersecurity")
]


def get_department(page):
    for start, end, department in department_ranges:
        if start <= page <= end:
            return department
    return "Unknown"


def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def extract_title(header):

    before_supervisor = re.split(
        r"(?im)^[ \t]*Supervisor[ \t]*:",
        header,
        maxsplit=1
    )[0].strip()

    paragraphs = re.split(
        r"\n[ \t]*\n",
        before_supervisor
    )

    title = clean_text(paragraphs[0])

    needs_review = (
        len(paragraphs) < 2 or not title
    )

    return title, needs_review


def remove_footer(text):

    pattern = (
        r"(?im)^[ \t]*Graduation Projects"
        r"[ \t]+2024[ \t]*[–—-][ \t]*2025"
        r"[ \t]*\d+[ \t]*$"
    )

    return re.sub(pattern, "", text).strip()


def split_abstract_keywords(text):

    text = remove_footer(text)

    keyword_pattern = (
        r"(?im)^[ \t]*Key[ -]?words?\b"
        r"[ \t]*[:：]?[ \t]*"
    )

    match = re.search(keyword_pattern, text)

    if match:
        abstract = text[:match.start()]
        keywords = text[match.end():]
    else:
        abstract = text
        keywords = ""

    return (
        clean_text(abstract),
        clean_text(keywords)
    )


# Build the structured dataset

projects = []

for idx, row in raw_df.iterrows():

    title, needs_review = extract_title(
        row["header_text"]
    )

    abstract, keywords = split_abstract_keywords(
        row["abstract_text"]
    )

    projects.append({
        "project_id": f"P{idx + 1:03d}",
        "title": title,
        "abstract": abstract,
        "keywords": keywords,
        "department": get_department(
            row["source_page"]
        ),
        "year": "2024-2025",
        "source_page": row["source_page"],
        "needs_review": needs_review
    })


projects_df = pd.DataFrame(projects)

print("STRUCTURED DATASET CREATED")
print("=" * 50)

print("Total projects:", len(projects_df))
print("Columns:", projects_df.columns.tolist())

display(projects_df.head())


STRUCTURED DATASET CREATED
Total projects: 103
Columns: ['project_id', 'title', 'abstract', 'keywords', 'department', 'year', 'source_page', 'needs_review']


,project_id,title,abstract,keywords,department,year,source_page,needs_review
0,P001,Automatic Plant Watering and Tracking System,"Many people struggle with watering their plants regularly, which usually leads to plants becoming unhealthy or dying. To address this common issue, it is important to monitor basic factors such as soil moisture and temperature in order to determine the proper time and amount of water needed. Since manual watering is not always consistent or reliable, this automatic plant watering system is designed to assist gardeners by providing an accurate irrigation based on real time weather conditions and the specific water requirements of each plant. The system includes a monitoring mechanism that helps ensure proper and efficient water usage, reducing water waste and supporting healthy plant growth. Additionally, the application includes a built-in encyclopedia that offers detailed information about a variety of plant types, helping users to understand their plant’s water needs, caring details and other useful facts. This solution makes it easier to take care for plants, especially when they are located away from owners.","Moisturem Air humidity, AWTS system, Waterin, Plants .",Computer Science,2024-2025,5,False
1,P002,CourseAssist: CLO Performance Assistant “E-CPA”,"The rapid evolution of educational technology necessitates innovative approaches to enhance the assessment of student performance and course effectiveness. Our project addresses a significant issue present when faculty members spend an excessive amount of time completing the necessary documentation to achieve course objectives. To tackle this problem, we developed ""Alama,"" an automated mobile application that streamlines the traditional Excel data entry process. This application calculates Course Learning Outcomes (CLOs) and Program Learning Outcomes (PLOs) automatically, generating performance reports in moments rather than relying on manual entry. Alama not only mitigates the inefficiencies of conventional assessment methods, which can lead to inaccuracies and impede effective decision-making, but also integrates both quantitative and qualitative data. This holistic approach enhances the evaluation of educational processes, ultimately improving the learning experience for students. In conclusion, the implementation of Alama presents a significant advancement in academic assessment, offering both advantages—such as time savings and enhanced accuracy—and potential drawbacks, including the initial learning curve for faculty. Overall, Alama holds the promise of transforming educational assessment practices, leading to better educational outcomes.","Program Learning Outcomes, Course Learning Outcomes, Performance Assistant, Formative, Summative",Computer Science,2024-2025,6,False
2,P003,Mobile Application for Deaf and Mute individuals. (WASIF),"Our System, Wasif, offers a communication application for deaf and mute individuals, we aim to develop cross-platform application that bridges these communication gaps by translating spoken language into sign language symbols and written text, while converting sign language gestures into audible speech. Wasif application analyzes hand and facial movements in real-time, ensuring accurate and efficient communication. Built using robust frameworks which is TensorFlow, and OpenCV, seeks to empower deaf and mute individuals to fully engage in all aspects of life, it is have been validated through rigorous testing with target user groups, ensuring that it meets the real-world needs.","Arabic Sign Language, Sign Language Gestures into Audial Speech, Real-time Facial and Hand Movement Tracking, Deaf and mute.",Computer Science,2024-2025,7,False
3,P004,Pet Care Application System (Anees),"Pets have always played a vital role in human companionship, offering emotional support, reducing stress, and enhancing overall well-being. As pet ownership continues to grow in Saudi Arabia, so there is a high demand for accessib


## 6. Data Quality Validation

Evaluate the quality and consistency of the extracted dataset before exporting it.

Validation includes checking missing values, duplicate titles, unknown departments, keyword coverage, abstract lengths, and project distribution across departments.

Project titles at department boundaries are inspected to verify correct classification.

Potential extraction issues and missing keywords are also reviewed manually.

 [link text](https://)

In [ ]:

# ============================================================
# 6. DATA QUALITY VALIDATION
# Origin - Graduation Projects Dataset (2024-2025)
# ============================================================

import re
import pandas as pd
from IPython.display import display

print("DATA QUALITY VALIDATION")
print("=" * 55)

# ------------------------------------------------------------
# 1. Dataset Overview
# ------------------------------------------------------------

print("\n1. DATASET OVERVIEW")
print("-" * 55)

print("Total PDF pages:", len(doc))
print("Total projects:", len(projects_df))
print("Total columns:", len(projects_df.columns))

print("\nDataset columns:")
print(projects_df.columns.tolist())


# ------------------------------------------------------------
# 2. Missing Values and Duplicates
# ------------------------------------------------------------

print("\n2. MISSING VALUES AND DUPLICATES")
print("-" * 55)

print(
    "Missing titles:",
    projects_df["title"].eq("").sum()
)

print(
    "Missing abstracts:",
    projects_df["abstract"].eq("").sum()
)

print(
    "Duplicate project IDs:",
    projects_df["project_id"].duplicated().sum()
)

print(
    "Duplicate titles:",
    projects_df["title"].duplicated().sum()
)

print(
    "Duplicate source pages:",
    projects_df["source_page"].duplicated().sum()
)

print(
    "Unknown departments:",
    projects_df["department"].eq("Unknown").sum()
)

print(
    "Titles requiring review:",
    projects_df["needs_review"].sum()
)


# ------------------------------------------------------------
# 3. Department Distribution
# ------------------------------------------------------------

print("\n3. PROJECTS BY DEPARTMENT")
print("-" * 55)

expected_counts = {
    "Computer Science": 33,
    "Artificial Intelligence": 12,
    "Information Systems": 23,
    "Data Science and Analysis": 10,
    "Information Technology": 17,
    "Cybersecurity": 8
}

actual_counts = (
    projects_df["department"]
    .value_counts()
    .to_dict()
)

department_summary = pd.DataFrame({
    "department": list(expected_counts.keys()),
    "expected": list(expected_counts.values())
})

department_summary["actual"] = (
    department_summary["department"]
    .map(actual_counts)
    .fillna(0)
    .astype(int)
)

department_summary["match"] = (
    department_summary["expected"]
    == department_summary["actual"]
)

display(department_summary)


# ------------------------------------------------------------
# 4. Keywords Validation
# ------------------------------------------------------------

print("\n4. KEYWORDS VALIDATION")
print("-" * 55)

print(
    "Projects with keywords:",
    projects_df["keywords"].ne("").sum()
)

print(
    "Projects without keywords:",
    projects_df["keywords"].eq("").sum()
)

print("\nProjects without keywords:")

display(
    projects_df.loc[
        projects_df["keywords"].eq(""),
        ["project_id", "title", "source_page"]
    ]
)


# ------------------------------------------------------------
# 5. Abstract Length Statistics
# ------------------------------------------------------------

print("\n5. ABSTRACT LENGTH STATISTICS")
print("-" * 55)

word_counts = (
    projects_df["abstract"]
    .str.split()
    .str.len()
)

print("Minimum words:", word_counts.min())
print("Maximum words:", word_counts.max())
print("Average words:", round(word_counts.mean(), 1))


# ------------------------------------------------------------
# 6. Department Boundary Check
# ------------------------------------------------------------

print("\n6. DEPARTMENT BOUNDARY CHECK")
print("-" * 55)

# Updated page numbers for the 115-page PDF

boundary_pages = [
    5, 37,       # Computer Science
    39, 50,      # Artificial Intelligence
    52, 74,      # Information Systems
    76, 85,      # Data Science and Analysis
    87, 103,     # Information Technology
    105, 112     # Cybersecurity
]

display(
    projects_df.loc[
        projects_df["source_page"].isin(
            boundary_pages
        ),
        [
            "project_id",
            "title",
            "department",
            "source_page"
        ]
    ]
)


# ------------------------------------------------------------
# 7. Remaining Keyword Mentions
# ------------------------------------------------------------

print("\n7. KEYWORD MENTIONS WITHIN ABSTRACTS")
print("-" * 55)

keyword_mask = (
    projects_df["abstract"].str.contains(
        r"\bkeywords?\b",
        case=False,
        regex=True,
        na=False
    )
)

print(
    "Abstracts mentioning keywords:",
    keyword_mask.sum()
)

for _, row in projects_df[keyword_mask].iterrows():

    print("\nProject:", row["title"])
    print("Page:", row["source_page"])

    for match in re.finditer(
        r"\bkeywords?\b",
        row["abstract"],
        re.I
    ):
        start = max(0, match.start() - 100)
        end = min(
            len(row["abstract"]),
            match.end() + 100
        )

        print("Context:")
        print(row["abstract"][start:end])


# ------------------------------------------------------------
# 8. Suspicious Records Check
# ------------------------------------------------------------

print("\n8. POTENTIALLY SUSPICIOUS RECORDS")
print("-" * 55)

suspicious = projects_df[
    (word_counts < 40)
    |
    projects_df["abstract"].str.contains(
        r"Graduation Projects\s*2024|Supervisor\s*:",
        case=False,
        regex=True,
        na=False
    )
]

print("Suspicious records:", len(suspicious))

display(
    suspicious[
        ["project_id", "title", "source_page"]
    ]
)


# ------------------------------------------------------------
# 9. Final Automated Validation
# ------------------------------------------------------------

print("\n9. FINAL AUTOMATED VALIDATION")
print("-" * 55)

assert len(doc) == 115, (
    "Incorrect PDF version. Expected 115 pages."
)

assert len(projects_df) == 103, (
    f"Expected 103 projects, found {len(projects_df)}."
)

assert len(projects_df) == len(raw_df), (
    "Mismatch between raw and structured records."
)

assert projects_df["project_id"].is_unique, (
    "Duplicate project IDs detected."
)

assert projects_df["source_page"].is_unique, (
    "Duplicate source pages detected."
)

assert projects_df["title"].ne("").all(), (
    "Missing project titles detected."
)

assert projects_df["abstract"].ne("").all(), (
    "Missing project abstracts detected."
)

assert projects_df["department"].ne("Unknown").all(), (
    "Unknown departments detected."
)

assert actual_counts == expected_counts, (
    "Department counts do not match expected values."
)

print("\nALL AUTOMATED VALIDATION CHECKS PASSED!")
print("Total validated projects:", len(projects_df))
print("Total departments:", len(expected_counts))
print("Dataset is ready for final manual review.")


DATA QUALITY VALIDATION

1. DATASET OVERVIEW
-------------------------------------------------------
Total PDF pages: 115
Total projects: 103
Total columns: 8

Dataset columns:
['project_id', 'title', 'abstract', 'keywords', 'department', 'year', 'source_page', 'needs_review']

2. MISSING VALUES AND DUPLICATES
-------------------------------------------------------
Missing titles: 0
Missing abstracts: 0
Duplicate project IDs: 0
Duplicate titles: 0
Duplicate source pages: 0
Unknown departments: 0
Titles requiring review: 0

3. PROJECTS BY DEPARTMENT
-------------------------------------------------------


,department,expected,actual,match
0,Computer Science,33,33,True
1,Artificial Intelligence,12,12,True
2,Information Systems,23,23,True
3,Data Science and Analysis,10,10,True
4,Information Technology,17,17,True
5,Cybersecurity,8,8,True



4. KEYWORDS VALIDATION
-------------------------------------------------------
Projects with keywords: 102
Projects without keywords: 1

Projects without keywords:


,project_id,title,source_page
45,P046,SaudiWay,52



5. ABSTRACT LENGTH STATISTICS
-------------------------------------------------------
Minimum words: 61
Maximum words: 277
Average words: 171.6

6. DEPARTMENT BOUNDARY CHECK
-------------------------------------------------------


,project_id,title,department,source_page
0,P001,Automatic Plant Watering and Tracking System,Computer Science,5
32,P033,RASID | Road defect detection and classification,Computer Science,37
33,P034,Visualize,Artificial Intelligence,39
44,P045,Thaqib | Enhancing Critical Thinking with Multimodal Generative AI,Artificial Intelligence,50
45,P046,SaudiWay,Information Systems,52
67,P068,MALI – Expense Tracker Application,Information Systems,74
68,P069,Bridging Communication Gaps: A system for Arabic sign language translation,Data Science and Analysis,76
77,P078,AI-Driven Serious Games for ADHD Treatment,Data Science and Analysis,85
78,P079,A Software-Defined Networking Approach to Link Failure Solutions in Cloud Environments,Information Technology,87
94,P095,Image Forgery Detection Website: ForgeFinder,Information Technology,103



7. KEYWORD MENTIONS WITHIN ABSTRACTS
-------------------------------------------------------
Abstracts mentioning keywords: 1

Project: AI-Powered Platform for Monitoring and Analyzing Consumer Sentiments in the Cosmetics Industry
Page: 78
Context:
nsights. The platform includes an integrated search engine that allows users to find products using keywords or hashtags, connecting directly to a text-based social media platform, such as Twitter. The system

8. POTENTIALLY SUSPICIOUS RECORDS
-------------------------------------------------------
Suspicious records: 0


,project_id,title,source_page



9. FINAL AUTOMATED VALIDATION
-------------------------------------------------------

ALL AUTOMATED VALIDATION CHECKS PASSED!
Total validated projects: 103
Total departments: 6
Dataset is ready for final manual review.



## 7. Final Dataset Adoption and Validation

The initial PDF extraction pipeline was used to generate a preliminary graduation projects dataset.

After reviewing and comparing the extracted records against the original PDF, the dataset was further refined using Codex-assisted data processing.

The refined dataset was selected as the final version based on the team's validation and comparison process.

The finalized file, `projects_clean.csv`, is used as the authoritative dataset for all subsequent Origin experiments.

This section loads and validates the approved dataset without regenerating or overwriting it.
